# MultiView

When visualizing different properties, various visual attributes such as x & y position, color, size, shape, etc. are often used. However, with an increasing number of encoding channels, a visualization can quickly become confusing and difficult to read. An alternative to “overloading” a single visualization is to combine several visualizations in such a way that a quick comparison is possible.

This notebook describes various operations for combining multiple views:

* **layer:** Placing compatible visualizations on top of each other,
* **facet:** Data is split into multiple visualizations organized in rows or columns,
* **concatenate:** Positioning of any visualizations in a common layout, and
* **repeat:** apply a visualization specification to multiple data



In [49]:
import pandas as pd
import altair as alt

In [50]:
weather = 'https://cdn.jsdelivr.net/npm/vega-datasets@1/data/weather.csv'

In [51]:
df = pd.read_csv(weather)
df

,location,date,precipitation,temp_max,temp_min,wind,weather
0,Seattle,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,Seattle,2012-01-02,10.9,10.6,2.8,4.5,rain
2,Seattle,2012-01-03,0.8,11.7,7.2,2.3,rain
3,Seattle,2012-01-04,20.3,12.2,5.6,4.7,rain
4,Seattle,2012-01-05,1.3,8.9,2.8,6.1,rain
...,...,...,...,...,...,...,...
2917,New York,2015-12-27,2.0,17.2,8.9,5.5,fog
2918,New York,2015-12-28,1.3,8.9,1.7,6.3,snow
2919,New York,2015-12-29,16.8,9.4,1.1,5.3,fog
2920,New York,2015-12-30,9.4,10.6,5.0,3.0,fog


## Layer

One of the most common ways to combine multiple visualizations is to superimpose visual elements. If the underlying scale ranges are compatible, they can be merged to use common axes.

Output of minimum and maximum temperatures:

In [52]:
c = alt.Chart(weather).mark_area().encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_max):Q'),
  alt.Y2('average(temp_min):Q')
)
c

alt.Chart(...)

Let’s subdivide the data by location using a color encoding, while also adjusting the mark opacity to accommodate overlapping areas:


In [53]:
alt.Chart(weather).mark_area(opacity=0.3).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_max):Q'),
  alt.Y2('average(temp_min):Q'),
  alt.Color('location:N')
)

alt.Chart(...)

Let’s create a line chart showing the average temperature midpoint. We’ll use a calculate transform to compute the midpoints between the minimum and maximum daily temperatures:

In [54]:
alt.Chart(weather).mark_line().transform_calculate(
  temp_mid='(+datum.temp_min + +datum.temp_max) / 2' #As we are loading the data directly from a CSV file without any special parsing instructions, the temperature values may be internally represented as string values. Adding the + in front of the value forces it to be treated as a number
).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_mid):Q'),
  alt.Color('location:N')
)

alt.Chart(...)

We’d now like to combine these charts by layering the midpoint lines over the range areas. Using the syntax *chart1 + chart2*, we can specify that we want a new layered chart in which chart1 is the first layer and chart2 is a second layer drawn on top:

In [55]:
tempMinMax = alt.Chart(weather).mark_area(opacity=0.3).encode(
  alt.X('month(date):T', title=None, axis=alt.Axis(format='%b')),
  alt.Y('average(temp_max):Q', title='Avg. Temperature °C'), # Anpassung der Achsenbeschriftung
  alt.Y2('average(temp_min):Q'),
  alt.Color('location:N')
)

tempMid = alt.Chart(weather).mark_line().transform_calculate(
  temp_mid='(+datum.temp_min + +datum.temp_max) / 2'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_mid):Q'),
  alt.Color('location:N')
)

tempMinMax + tempMid

alt.LayerChart(...)

# A few more cities... World Dataset

In [56]:
#!pip install meteostat

#### this may take a while. Fetch data from servers
best: download once, save data to CSV

In [ ]:
from datetime import date
import meteostat as ms
import pandas as pd

# Define time range (matching vega-datasets 2012-2015)
start = date(2023, 1, 1)
end = date(2025, 12, 31)

# Pick any cities using (latitude, longitude)
cities = {
    "Vienna": ms.Point(48.2082, 16.3738),
    "Tokyo": ms.Point(35.6762, 139.6503),
    "London": ms.Point(51.5072, -0.1276),
    "Sydney": ms.Point(-33.8688, 151.2093),
    "Buenos Aires": ms.Point(-34.603722, -58.381592),
}

frames = []
for city_name, point in cities.items():
  # 1. Find nearby weather stations for the point
  stations = ms.stations.nearby(point, limit=4)

  # 2. Fetch daily data and interpolate for the exact coordinates
  ts = ms.daily(stations, start, end)
  df = ms.interpolate(ts, point).fetch().reset_index()

  df["location"] = city_name
  frames.append(df)


In [ ]:

# Combine dataframes and rename columns
weather = pd.concat(frames, ignore_index=True).rename(
    columns={
        "time": "date",
        "prcp": "precipitation",
        "tmax": "temp_max",
        "tmin": "temp_min",
        "wspd": "wind",
        "snwd": "snow"
    }
)[["location", "date", "precipitation", "temp_max", "temp_min", "wind", "snow"]]

weather.describe()

In [ ]:
#add nice descriptions
weather.attrs["column_descriptions"] = {
    "location": "City name (category)",
    "date": "Observation date (YYYY-MM-DD)",
    "precipitation": "Total daily precipitation (mm)",
    "temp_max": "Maximum daily air temperature at 2m (°C)",
    "temp_min": "Minimum daily air temperature at 2m (°C)",
    "wind": "Maximum daily wind speed at 10m (km/h)",
    "snow": "Snow Depth [cm]"
}

In [ ]:
#save to csv
weather.to_csv("weather.csv")

In [ ]:
import pandas as pd
import altair as alt
#### much faster: load saved csv
weather = pd.read_csv("weather.csv", index_col="Unnamed: 0")
weather["date"] = pd.to_datetime(weather["date"])

In [ ]:
weather

In [ ]:
axis_labels = {
    "location": "City",
    "date": "Date",
    "precipitation": "Precipitation [mm]",
    "temp_max": "Max Temperature [°C]",
    "temp_min": "Min Temperature [°C]",
    "wind": "Max Wind Speed [km/h]",
    "snow": "Snow Depth [cm]"
}

In [ ]:
# Extract ISO week number and compute daily mean
weather["week"] = weather["date"].dt.isocalendar().week.astype(int)
weather["temp_avg"] = (weather["temp_max"] + weather["temp_min"]) / 2

weekly = (
    weather.groupby(["location", "week"], as_index=False)
    .agg(
        temp_min=("temp_min", "mean"),
        temp_max=("temp_max", "mean"),
        temp_avg=("temp_avg", "mean"),
        precipitation=("precipitation", "mean"),
    )
    .query("week <= 52")
    .copy()
)

# Convert week number to a reference date in a single year for clean month ticks
weekly["week_date"] = pd.to_datetime("2012-01-01") + pd.to_timedelta((weekly["week"] - 1) * 7, unit="D")


In [ ]:

# Build layered chart
base = alt.Chart(weekly).encode(
    x=alt.X("week:Q", title="Week of Year", scale=alt.Scale(domain=[1, 52])),
    color=alt.Color("location:N", title="City"),
)

band = base.mark_area(opacity=0.2).encode(
    y=alt.Y("temp_min:Q", title="Temperature (°C)"), y2="temp_max:Q"
)

line = base.mark_line(strokeWidth=2.5).encode(y="temp_avg:Q")

(band + line).properties(width=650, height=350)


In [ ]:
# Build layered chart with month labels on the x-axis
base = alt.Chart(weekly).encode(
    x=alt.X(
        "week_date:T",
        title="Month",
        axis=alt.Axis(format="%b", tickCount="month", labelAlign="left"),
    ),
    # Force full opacity in the legend regardless of mark opacity
    color=alt.Color(
        "location:N", title="City", legend=alt.Legend(symbolOpacity=1)
    )
)

band = base.mark_area(opacity=0.2).encode(
    y=alt.Y("temp_min:Q", title="Temperature [°C]"), y2="temp_max:Q"
)

line = base.mark_line(strokeWidth=2.5).encode(y="temp_avg:Q")

(band + line).properties(width=650, height=350)


In [ ]:
alt.layer(
    alt.Chart().mark_area(opacity=0.25).encode(y="temp_min:Q", y2="temp_max:Q"),
    alt.Chart().mark_line(strokeWidth=2.5).encode(y="temp_avg:Q"),
    data=weekly,
).encode(
    x=alt.X(
        "week_date:T",
        title="Month",
        axis=alt.Axis(format="%b", tickCount="month", labelAlign="left"),
    ),
    y=alt.Y(title="Temperature [°C]"),
    color=alt.Color("location:N", legend=None),
).properties(
    width=220, height=200
).facet(
    facet=alt.Facet("location:N", title="City"), columns=5
)


In [ ]:
# Filter to Vienna only
vienna = weekly[weekly["location"] == "Vienna"]

temps = alt.layer(
    alt.Chart().mark_area(opacity=0.25).encode(y="temp_min:Q", y2="temp_max:Q"),
    alt.Chart().mark_line(strokeWidth=2.5).encode(y="temp_avg:Q"),
    data=vienna
).encode(
    x=alt.X(
        "week_date:T",
        title="Month",
        axis=alt.Axis(format="%b", tickCount="month", labelAlign="left"),
    ),
    y=alt.Y(title="Temperature [°C]"),
    color=alt.Color("location:N", legend=None),
).properties(
    width=700, height=500
)

rains = (
    alt.Chart(vienna).mark_bar(opacity=0.25, size=6, color="#4c78a8")
    .encode(
        x=alt.X(
                "week_date:T",
                title="Month",
                axis=alt.Axis(format="%b", tickCount="month", labelAlign="left"),
        ),
        y=alt.Y( "precipitation:Q", title="Precipitation [mm/day]")
        
    )
)

(rains + temps).resolve_scale(y="independent")


In [ ]:
# Filter to Vienna only
vienna = weekly#[weekly["location"] == "Vienna"]

temps = alt.layer(
    alt.Chart().mark_area(opacity=0.25).encode(y="temp_min:Q", y2="temp_max:Q"),
    alt.Chart().mark_line(strokeWidth=2.5).encode(y="temp_avg:Q"),
    data=vienna
).encode(
    x=alt.X(
        "week_date:T",
        title="Month",
        axis=alt.Axis(format="%b", tickCount="month", labelAlign="left"),
    ),
    y=alt.Y(title="Temperature [°C]"),
    color=alt.Color("location:N", legend=None),
).properties(
    width=700, height=500
)

rains = (
    alt.Chart(vienna).mark_bar(opacity=0.25, size=6, color="#4c78a8")
    .encode(
        x=alt.X(
                "week_date:T",
                title="Month",
                axis=alt.Axis(format="%b", tickCount="month", labelAlign="left"),
        ),
        y=alt.Y( "precipitation:Q", title="Precipitation [mm/day]")
        
    )
)
#facet with second axis doesn't work
(rains + temps).resolve_scale(y="independent").facet(
    facet=alt.Facet("location:N", title="City"), columns=5
)


In [ ]:
charts = []
for city in list(cities.keys()):
    
    df = weekly[weekly["location"] == city]

    temps = alt.layer(
        alt.Chart().mark_area(opacity=0.25).encode(y="temp_min:Q", y2="temp_max:Q"),
        alt.Chart().mark_line(strokeWidth=2.5).encode(y="temp_avg:Q"),
        data=df
    ).encode(
        x=alt.X("week_date:T", title="Month", axis=alt.Axis(format="%b", tickCount="month", labelAlign="left")),
        y=alt.Y(title="Temperature [°C]"),
        color=alt.Color("location:N", legend=None),
    ).properties(
        width=200, height=200
    )

    rains = (
        alt.Chart(df).mark_bar(opacity=0.25, size=6)
        .encode(
            x=alt.X("week_date:T", title="Month", axis=alt.Axis(format="%b", tickCount="month", labelAlign="left")),
            y=alt.Y( "precipitation:Q", title="Precipitation [mm/day]"),
            color=alt.Color("location:N", legend=None)
        )
    )

    #independent y-axis for the first chart:
    charts.append((rains + temps).resolve_scale(y="independent"))
    
alt.hconcat(*charts)

In [ ]:
charts = []
for city in list(cities.keys()):
    
    df = weekly[weekly["location"] == city]

    temps = alt.layer(
        alt.Chart().mark_area(opacity=0.25).encode(y="temp_min:Q", y2="temp_max:Q"),
        alt.Chart().mark_line(strokeWidth=2.5).encode(y="temp_avg:Q"),
        data=df
    ).encode(
        x=alt.X("week_date:T", title="Month", axis=alt.Axis(format="%b", tickCount="month", labelAlign="left")),
        y=alt.Y(title="Temperature [°C]", scale=alt.Scale(domain=[-10, 35]),),
        color=alt.Color("location:N", legend=None),
    ).properties(
        width=200, height=200
    )

    rains = (
        alt.Chart(df).mark_bar(opacity=0.25, size=6)
        .encode(
            x=alt.X("week_date:T", title="Month", axis=alt.Axis(format="%b", tickCount="month", labelAlign="left")),
            y=alt.Y( "precipitation:Q", scale=alt.Scale(domain=[0, 40]), title="Precipitation [mm/day]"),
            color=alt.Color("location:N", legend=None)
        )
    )

    #independent y-axis for the first chart:
    charts.append((rains + temps).resolve_scale(y="independent"))
    
alt.hconcat(*charts)

In [ ]:
# Top row: Temperature ribbon + average line
temp_chart = (
    alt.layer(
        alt.Chart()
        .mark_area(opacity=0.25)
        .encode(y="temp_min:Q", y2="temp_max:Q"),
        alt.Chart().mark_line(strokeWidth=2.5).encode(y="temp_avg:Q"),
        data=weekly,
    )
    .encode(
        x=alt.X(
            "week_date:T",
            title=None,  # Hide x-axis title on top row to avoid duplication
            axis=alt.Axis(format="%b", tickCount="month", labelAlign="left"),
        ),
        y=alt.Y(title="Temperature [°C]"),
        color=alt.Color("location:N", legend=None),
    )
    .properties(width=220, height=180)
    .facet(facet=alt.Facet("location:N", title="City"), columns=5)
)

# Bottom row: Average daily precipitation bars (4 columns)
rain_chart = (
    alt.Chart(weekly)
    .mark_bar(opacity=0.75)
    .encode(
        x=alt.X(
            "week_date:T",
            title="Month",
            axis=alt.Axis(format="%b", tickCount="month", labelAlign="left"),
        ),
        y=alt.Y("precipitation:Q", title="Precipitation [mm/day]"),
        color=alt.Color("location:N", legend=None),
        facet=alt.Facet("location:N", title=None, columns=5),
    )
    .properties(width=220, height=120)
)

(temp_chart & rain_chart)


In [ ]:
import altair as alt
import pandas as pd

# 1. Shared scale with nice=False so bars do not expand the start/end dates
x_scale = alt.Scale(domain=["2012-01-01", "2012-12-25"], nice=False)
x_axis = alt.Axis(format="%b", tickCount="month", labelAlign="left")

# 2. Lock identical y-axis label width (minExtent) so left plot edges match
y_temp = alt.Y(title="Temperature [°C]", axis=alt.Axis(minExtent=35))
y_rain = alt.Y(
    "precipitation:Q",
    title="Precipitation [mm/day]",
    axis=alt.Axis(minExtent=35),
)

# 3. Top row: Temperature (built with .facet())
temp_chart = (
    alt.layer(
        alt.Chart()
        .mark_area(opacity=0.25)
        .encode(y="temp_min:Q", y2="temp_max:Q"),
        alt.Chart().mark_line(strokeWidth=2.5).encode(y="temp_avg:Q"),
        data=weekly,
    )
    .encode(
        x=alt.X("week_date:T", title=None, scale=x_scale, axis=x_axis),
        y=y_temp,
        color=alt.Color("location:N", legend=None),
    )
    .properties(width=220, height=180)
    .facet(facet=alt.Facet("location:N", title="City"), columns=5)
)

# 4. Bottom row: Precipitation (also built with .facet() to match layout structure)
rain_chart = (
    alt.Chart(weekly)
    .mark_bar(opacity=0.75, size=3)
    .encode(
        x=alt.X("week_date:T", title="Month", scale=x_scale, axis=x_axis),
        y=y_rain,
        color=alt.Color("location:N", legend=None),
    )
    .properties(width=220, height=120)
    .facet(
        facet=alt.Facet(
            "location:N", title=None, header=alt.Header(labels=False)
        ),
        columns=5,
    )
)

# 5. Stack vertically with unified layout bounds
alt.vconcat(temp_chart, rain_chart, spacing=10).resolve_scale(x="shared")


## Facet

Faceting involves subdividing a dataset into groups and creating a separate plot for each group (using row or column). We’ll first review those channels and then show how they are instances of the more general facet operator.

Let’s start with a basic histogram of maximum temperature values in Seattle:

In [ ]:
alt.Chart(weather).mark_bar().transform_filter(
  'datum.location == "Seattle"'
).encode(
  alt.X('temp_max:Q', bin=True, title='Temperature (°C)'),
  alt.Y('count():Q')
)

Let’s use the *column* encoding channel to facet the data by weather type. We can also use color as a redundant encoding, using a customized color range:

In [ ]:
colors = alt.Scale(
  domain=['drizzle', 'fog', 'rain', 'snow', 'sun'],
  range=['#aec7e8', '#c7c7c7', '#1f77b4', '#9467bd', '#e7ba52']
)

alt.Chart(weather).mark_bar().transform_filter(
  'datum.location == "Seattle"'
).encode(
  alt.X('temp_max:Q', bin=True, title='Temperature (°C)'),
  alt.Y('count():Q'),
  alt.Color('weather:N', scale=colors),
  alt.Column('weather:N')
).properties(
  width=150,
  height=150
)

In addition to *row* and *column* encoding channels within a chart definition, we can take a basic chart definition and apply faceting using an explicit *facet* operator.

In [ ]:
colors = alt.Scale(
  domain=['drizzle', 'fog', 'rain', 'snow', 'sun'],
  range=['#aec7e8', '#c7c7c7', '#1f77b4', '#9467bd', '#e7ba52']
)

alt.Chart().mark_bar().encode(
  alt.X('temp_max:Q', bin=True, title='Temperature (°C)'),
  alt.Y('count():Q'),
  alt.Color('weather:N', scale=colors)
).properties(
  width=150,
  height=150
).facet( # invoke the facet method, passing in the data and specifying that we facet into columns according to the weather field
  data=weather,
  column='weather:N' # the facet method accepts both row and column arguments
).transform_filter(
 'datum.location == "Seattle"'
)


Given all the extra code above, why would we want to use an explicit facet operator? For basic charts, we should certainly use the column or row encoding channels if we can. However, using the facet operator explicitly is useful if we want to facet composed views, such as layered charts.

**Column version:**

In [ ]:
tempMinMax = alt.Chart().mark_area(opacity=0.3).encode(
  alt.X('month(date):T', title=None, axis=alt.Axis(format='%b')),
  alt.Y('average(temp_max):Q', title='Avg. Temperature (°C)'),
  alt.Y2('average(temp_min):Q'),
  alt.Color('location:N')
)

tempMid = alt.Chart().mark_line().transform_calculate(
  temp_mid='(+datum.temp_min + +datum.temp_max) / 2'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_mid):Q'),
  alt.Color('location:N')
)

alt.layer(tempMinMax, tempMid).facet(
  data=weather,
  column='location:N'
)

**Faceting version:**

The faceted charts we have seen so far use the same axis scale domains across the facet cells. This default of using shared scales and axes helps aid accurate comparison of values. However, in some cases you may wish to scale each chart independently, for example if the range of values in the cells differs significantly.

In [ ]:
tempMinMax = alt.Chart().mark_area(opacity=0.3).encode(
  alt.X('month(date):T', title=None, axis=alt.Axis(format='%b')),
  alt.Y('average(temp_max):Q', title='Avg. Temperature (°C)'),
  alt.Y2('average(temp_min):Q'),
  alt.Color('location:N')
)

tempMid = alt.Chart().mark_line().transform_calculate(
  temp_mid='(+datum.temp_min + +datum.temp_max) / 2'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_mid):Q'),
  alt.Color('location:N')
)

alt.layer(tempMinMax, tempMid).facet(
  data=weather,
  column='location:N'
).resolve_axis(y='independent') # Alternative: .resolve_scale(y='independent') skaliert die y-Achse unabhängig von den beiden Visualisierungen

## Concatenate

Faceting creates small multiple plots that show separate subdivisions of the data. However, we might wish to create a multi-view display with different views of the same dataset (not subsets) or views involving different datasets.

Altair provides concatenation operators to combine arbitrary charts into a composed chart. The *hconcat* operator (shorthand | ) performs horizontal concatenation, while the *vconcat* operator (shorthand &) performs vertical concatenation.

In [ ]:
base = alt.Chart(weather).mark_line().encode(
  alt.X('month(date):T', title=None),
  color='location:N'
).properties(
  width=240,
  height=180
)

temp = base.encode(alt.Y('average(temp_max):Q'))
precip = base.encode(alt.Y('average(precipitation):Q'))
wind = base.encode(alt.Y('average(wind):Q'))

temp | precip | wind

## Repeat

The concatenation operators above are quite general, allowing arbitrary charts to be composed. Nevertheless, the example above was still a bit verbose: we have three very similar charts, yet have to define them separately and then concatenate them.

For cases where only one or two variables are changing, the repeat operator provides a convenient shortcut for creating multiple charts. Given a template specification with some free variables, the *repeat* operator will then create a chart for each specified assignment to those variables.

In [ ]:
alt.Chart(weather).mark_line().encode(
  alt.X('month(date):T',title=None),
  alt.Y(alt.repeat('column'), aggregate='average', type='quantitative'),
  color='location:N'
).properties(
  width=240,
  height=180
).repeat( # invoke the repeat method to pass in data field names for each column
  column=['temp_max', 'precipitation', 'wind']
)

We can also use row and column repetition together! One common visualization for exploratory data analysis is the *scatter plot matrix (or SPLOM)*. Given a collection of variables to inspect, a SPLOM provides a grid of all pairwise plots of those variables, allowing us to assess potential associations.

In [ ]:
alt.Chart().mark_point(filled=True, size=15, opacity=0.5).encode(
  alt.X(alt.repeat('column'), type='quantitative'),
  alt.Y(alt.repeat('row'), type='quantitative')
).properties(
  width=150,
  height=150
).repeat(
  data=weather,
  row=['temp_max', 'precipitation', 'wind'],
  column=['wind', 'precipitation', 'temp_max']
).transform_filter(
  'datum.location == "Seattle"'
)